# 8. L2both per celltype

Part of the **[Fig. 1 chapter](fig1.md)** — see it for the panel-by-panel map. The first code cell sets `ENTEX_ROOT` and activates the no-overwrite guard (see the [Reproduction guide](../reproduce.md)). *Outputs shown are the author's original run.*


## 📥 Required input files

This notebook reads the following files (paths resolve from `ENTEX_ROOT`/`REF_ROOT`; the setup cell sets them). See the chapter's `inputs.md` for RAW-vs-derived tags.

- `f'{indir}npc.tsv'`  ·  _metadata_
- `f'{indir}../../L1color.tsv'`  ·  _metadata: color_
- `f'{indir}5kCG100k3C_summary.h5ad'`  ·  _joint summary obj_
- `f'{indir}celltype_annot.tsv'`  ·  _table_
- `f'{indir}L2/{l1_name}/5kCG100k3C_embed.h5ad'`  ·  _embedding h5ad_
- `f'{outdir}celltype_L2any/{ct_name}_mergeboth_rocpr80.hdf'`  ·  _other_
- `f'{indir}L2final_celltype_L2both_new.tsv'`  ·  _table_
- `f'{indir}../../tissuecolor.tsv'`  ·  _metadata: color_
- `f'{indir}../tissue/L1/{xx}/5kCG100k3C_embed.h5ad'`  ·  _embedding h5ad_
- `f'{indir}L2final_celltype_L2both_old.tsv'`  ·  _table_
- `f'{indir}celltype_annot_old.tsv'`  ·  _table_


In [ ]:
# === Reproduction setup — edit ENTEX_ROOT / REF_ROOT for your machine ===
import os, sys
ENTEX_ROOT = os.environ.get('ENTEX_ROOT', '/large_storage/zhoulab/zhoujt/project/ENTEx')
REF_ROOT   = os.environ.get('REF_ROOT',   '/large_storage/zhoulab/ref')
BOOK_ROOT  = os.environ.get('BOOK_ROOT',  f'{ENTEX_ROOT}/analysis/HumanCellEpigenomeAtlas')
sys.path.insert(0, BOOK_ROOT)
os.chdir(f'{ENTEX_ROOT}/analysis')   # original working directory
import repro_guard                    # no-overwrite guard (default: skip existing)

In [1]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
from glob import glob

import anndata
import scanpy as sc
import scanpy.external as sce
from sklearn.preprocessing import normalize
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score

from ALLCools.clustering import *
from ALLCools.plot import *
from ALLCools.integration.seurat_class import SeuratIntegration

mpl.style.use('default')
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42
mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

import warnings
warnings.filterwarnings("ignore")


/home/zhoujt/.conda/envs/analysis/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/zhoujt/.conda/envs/analysis/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/zhoujt/.conda/envs/analysis/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/zhoujt/.conda/envs/analysis/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/zhoujt/.conda/envs/analysis/lib/python3.10/site-packages/anndata/utils.py:429: FutureWarning: Importing 

In [2]:
indir = f'{ENTEX_ROOT}/clustering/merged/'
outdir = f'{ENTEX_ROOT}/analysis/pairwise_prediction/'


In [3]:
npc = pd.read_csv(f'{indir}npc.tsv', sep='\t', header=None, index_col=0, names=['npc_cg', 'npc_3c'])

In [10]:
def merge_both(l1_name, ct_name, label):
    global indir, outdir
    adata = anndata.read_h5ad(f'{indir}L2/{l1_name}/5kCG100k3C_embed.h5ad')[label.index].copy()
    npc_cg, npc_3c = npc.loc[l1_name].values    
    data_cg = normalize(adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}'][:, :npc_cg], axis=1)
    data_3c = normalize(adata.obsm[f'5kCG100k3C_u{npc_cg}pc{npc_3c}'][:, -npc_3c:], axis=1)
    clf = LogisticRegression(class_weight='balanced')
    skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=0)
    # label = adata.obs['leiden_init']
    # label = np.load(f'{outdir}{ct}_mergeany_rocpr.npy', allow_pickle=True)
    label = label.astype('category')
    leg = np.sort(label.unique())
    while len(leg)>1:
        count = label.astype(str).value_counts()
        score = np.zeros((2, 2, len(leg), len(leg)))
        for k,data in enumerate([data_cg, data_3c]):
            for i in np.arange(len(leg)-1):
                for j in np.arange(i+1, len(leg)):
                    selc = label.isin([leg[i], leg[j]])
                    if count.loc[leg[i]]<count.loc[leg[j]]:
                        label_dict = {leg[i]:1, leg[j]:0}
                    else:
                        label_dict = {leg[i]:0, leg[j]:1}
                    # selc = np.random.permutation(np.where(selc)[0])
                    X = data[selc]
                    y = label.values[selc].map(label_dict).astype(int)
                    pred = np.zeros(len(y))
                    for train_index, test_index in skf.split(X, y):
                        X_train, X_test, y_train, y_test = X[train_index], X[test_index], y[train_index], y[test_index]
                        pred[test_index] = clf.fit(X_train, y_train).predict_proba(X_test)[:,1]
                    score[0,k,i,j] = roc_auc_score(y, pred)
                    score[1,k,i,j] = average_precision_score(y, pred)
        score = np.min(np.min(score, axis=0), axis=0)
        min_score = np.min(score[np.triu_indices(len(leg), k=1)])
        if min_score>0.8:
            break
        else:
            xx,yy = np.where(score==min_score)
            xx, yy = leg[xx[0]], leg[yy[0]]
            # print(f'Merge {xx} {yy}')
            label[label==xx] = yy
            leg = np.sort(label.unique())

    count = label.astype(str).value_counts()
    labelmap = count.reset_index().reset_index().set_index('L2_final')['index'].to_dict()
    label_reorder = label.map(labelmap).astype(int)
    label_reorder.to_hdf(f'{outdir}celltype_L2any/{ct_name}_mergeboth_rocpr80.hdf', key='data')
    return len(leg)


In [8]:
tissue_meta = pd.read_csv(f'{indir}../../tissuecolor.tsv', sep='\t', header=0, index_col=0)


In [135]:
ds = 0.5
coord_base = 'tsne'
mpl.use('agg')

fig, axes = plt.subplots(6, 3, figsize=(7, 10), dpi=300, constrained_layout=True)
for i,xx in enumerate(tissue_meta.index):
    tissue_name = tissue_meta.loc[xx, 'tissue_annot']
    adata_tmp = anndata.read_h5ad(f'{indir}../tissue/L1/{xx}/5kCG100k3C_embed.h5ad')
    adata_tmp.obs['celltype'] = adata.obs['celltype_L2_both_abbr'].astype(str)
    ax = axes.flatten()[i]
    count = adata_tmp.obs['celltype'].value_counts()
    leg = count.index[count>=30]
    selc = adata_tmp.obs['celltype'].isin(leg)
    tmp = adata_tmp.obs.loc[selc].copy()
    _ = categorical_scatter(data=tmp,
                            ax=ax,
                            coord_base=coord_base,
                            hue='celltype',
                            # text_anno='celltype', 
                            s=ds,
                            labelsize=6,
                            max_points=None,
                            palette='tab20',
                            scatter_kws={'rasterized':True},
                            # legend_kws={'ncol':1}, 
                            # show_legend=True
                           )
    for yy in leg:
        x, y = np.mean(adata_tmp.obsm[f'X_{coord_base}'][adata_tmp.obs['celltype']==yy], axis=0)
        ax.text(x, y, yy, ha='center', va='center', fontsize=6)
        
    ax.set_title(tissue_name, fontsize=8)

for ax in axes.flatten()[tissue_meta.shape[0]:]:
    ax.axis('off')
    
fig.savefig('clustering_summary/tissue_celltype.pdf', transparent=True)
